# 装饰器 pytest用来增强函数的功能 不能更改函数的功能
# 执行逻辑：将被装饰函数整体打包，以参数的形式传给装饰器
# 执行顺序：有多个装饰器时，越靠近被装饰函数的，越先执行

In [ ]:
# 不传参数的装饰器

def my_decorator(func):
    func() # 1 加载装饰器的时候就执行了 类似于初始化 - 遇到@my_decorator 就执行
    def wrapper():
        print("Before calling the function.")
        func() #2 只有手动调用了say_hello() 才执行
        print("After called.")

    func() # 3 加载完装饰器阶段，定义完wrapper后再执行一次 - 遇到@my_decorator 就执行 但是比 #1 要晚执行
    return wrapper

@my_decorator
def say_hello():
    print("Hello")

print("-"* 10)
print(say_hello())

In [ ]:
def my_decorator(func):
    def wrapper():
        print("The first decorator.")
        print("Before calling the function.")
        func() 
        print("After called.")
    return wrapper

def my_decorator_a(func):

    def wrapper():
        print("The second decarator.")
        func()
        print("After called....")
    return wrapper

@my_decorator_a
@my_decorator
def say_hello():
    print("Hello") 

"""
-> say_hello = my_decorator_a(my_decorator(say_hello))
-> 多个装饰器：
   加载阶段：越靠近函数，越先执行包装逻辑。 
   调用阶段："外层包裹内层"
"""

print("-"* 10)
print(say_hello())

In [ ]:
# 带参数的装饰器

def my_decorator_with_args(*args,**kwargs): # 接收装饰器参数
    def my_decorator(func): # 接收被装饰的函数
    
        def wrapper(name):  # 执行逻辑，以及接收被装饰函数的参数
            print(f"装饰器参数 args: {args}")           # 打印装饰器参数
            print(f"装饰器参数 kwargs: {kwargs}") 

            print("Before calling the function.")
            res = func(name)   # 调用func() 才执行
            print("After called.")
            return res
        
        return wrapper
    
    return my_decorator


@my_decorator_with_args("new member",id=2,age=5)
def say_hello(name):
    print(f"Hello {name}")


say_hello("zhangsan")


总结：
1. 带参数的装饰器至少有三层嵌套，比不带参数的装饰器多一层。
2. 执行顺序：“从外到内” my_decorator_with_args -> my_decorator -> wrapper

In [ ]:
from decorator import divisible_timer

@divisible_timer(divisor=3)
def calculate_sum(num = 10000):
    i = 0
    s = 0
    while i <=num:
        s += i 
        i+=1
    return s


calculate_sum(888889)


结合之前加密和解密的实战，写一个自动对被装饰函数的参数进行加密的装饰器

1. 被装饰函数主要的作用是打印参数的内容
2. 装饰器接受一个参数，用于作为加密的key（密钥）
3. 装饰器分别拿到加密的key和被装饰函数的参数
4. 对被装饰函数的参数进行加密
5. 被装饰函数自动打印加密后的数据

In [ ]:
from cryptography.fernet import Fernet


default_key = Fernet.generate_key()
def encrypt_decorator(input_key = default_key):

    def decorator(func):

        def wrapper(msg):
            print(f"original message is {msg}")
            cipher = Fernet(input_key)
            encrypted = cipher.encrypt(msg.encode())
            # print(f"After encryted the message is {encrypted}")

            res = func(encrypted) # 将加密后的信息传回给func
            
            return res
        return wrapper
    return decorator


In [31]:
user_key = Fernet.generate_key()

@encrypt_decorator(input_key=user_key)
def say_hello_to_three_body(msg):
    print(f"Please do not answer !!!")
    print(f"The encrypted message is {msg}")

In [32]:
say_hello_to_three_body("世界属于三体")

original message is 世界属于三体
Please do not answer !!!
The encrypted message is b'gAAAAABqNSkSooH4g8puxAyathrWFVUwfydQZswNwreC_I4vUt-e-dMLgNAlI1YDiX_dRcy0wC5oq_sIcwQgRl-BQDNhTDx38tB6iDHU6jKhTVwRr94vU3I='
